# Скрининг риска анемии по фото глаза — v11

**Не диагноз.** Результат — шкала риска: **Норма / Лёгкая анемия / Умеренная анемия (или тяжелее) / Переснять фото**, и гемоглобин словами (в норме / немного снижен / низкий).

**Как запустить:** «Среда выполнения → Выполнить все». Первый запуск обучает модели один раз (≈3–6 мин, из них 1–2 мин скачивание Eyes-defy-anemia с Kaggle). Ячейка 3 скачает файл `anemia_models_v11.joblib` — в следующий раз перетащите его в панель «Файлы» слева (или включите Google Диск в ячейке 1), и модели загрузятся за секунды. Анализ одного фото — 1–3 секунды.

**Что делает система**
1. Находит глаз: по точкам лица (MediaPipe), если лицо в кадре; иначе считает снимок крупным планом глаза.
2. Находит конъюнктиву обученной моделью по разметке врачей Eyes-defy-anemia (не ручными порогами), склеру — по радужке и бесцветной светлой ткани рядом с ней.
3. Выбирает метод: видны обе — модель «конъюнктива + склера» (цвет века нормируется по белку глаза, это убирает влияние лампы); видна одна — модель только по ней; не видно ничего или фото размыто/тёмное/пересвечено — «Переснять».
4. Модель учится на каждом фото, перекрашенном под разные лампы и камеры, а классы уравнены: система не может «прятаться» в среднем классе.

**Точность** печатается в ячейке 3 и рисуется в ячейке 5 — только на пациентах, которых модель не видела, с матрицами ошибок по каждому классу.


In [ ]:
#@title ЯЧЕЙКА 1 — НАСТРОЙКА { display-mode: "form" }
#@markdown **Google Диск** — чтобы обученные модели сохранились и в следующий раз загрузились за секунды.
#@markdown Без Диска модели сохраняются файлом `anemia_models_v11.joblib`: ячейка 3 скачает его на компьютер,
#@markdown в следующий раз просто перетащите его в панель «Файлы» слева перед запуском.
Использовать_Google_Диск = False  #@param {type:"boolean"}

import os, sys, glob, shutil, subprocess, time, warnings, urllib.request
warnings.filterwarnings('ignore')
T0 = time.time()

# MediaPipe ставим без зависимостей: так он не трогает numpy/opencv Colab и не требует перезапуска среды
try:
    import mediapipe  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'mediapipe'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flatbuffers', 'absl-py'], check=False)
if not os.path.exists('face_landmarker.task'):
    try:
        urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/'
                                   'face_landmarker/float16/1/face_landmarker.task', 'face_landmarker.task')
    except Exception as e:
        print('⚠️ Модель поиска лица не скачалась (будет работать только режим «крупный план»):', str(e)[:100])

import numpy as np, pandas as pd, cv2, sklearn, joblib
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100

MODEL_NAME = 'anemia_models_v11.joblib'
WORK = 'anemia_v11'
ON_DRIVE = False
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK, ON_DRIVE = '/content/drive/MyDrive/anemia_v11', True
    except Exception as e:
        print('⚠️ Google Диск не подключился, работаю без него:', str(e)[:120])
os.makedirs(WORK, exist_ok=True)
MODEL_PATH = f'{WORK}/{MODEL_NAME}'
if not os.path.exists(MODEL_PATH) and os.path.exists(MODEL_NAME):      # файл моделей перетащили в «Файлы»
    shutil.copy(MODEL_NAME, MODEL_PATH)

print(f'✅ Ячейка 1 готова за {time.time() - T0:.0f} с · OpenCV {cv2.__version__} · scikit-learn {sklearn.__version__}')
print('   Модели:', 'найдены — обучение не понадобится' if os.path.exists(MODEL_PATH)
      else 'нет — ячейка 3 обучит их один раз (≈3–6 мин)')


In [ ]:
#@title ЯЧЕЙКА 2 — БИБЛИОТЕКА (только определения функций) { display-mode: "form" }
# Конвейер v11:
#   фото → поиск глаза (MediaPipe Face Landmarker; если лица нет — снимок считается крупным планом глаза)
#        → сегментация конъюнктивы (обученный попиксельный классификатор по разметке врачей)
#        → склера (белая, бесцветная ткань рядом с конъюнктивой; используется и как эталон белого)
#        → признаки цвета → модель «конъюнктива», «склера» или «обе вместе»
#        → класс: норма / лёгкая / умеренная и тяжелее / переснять
import os, glob, json, time, re, warnings
import numpy as np, cv2
warnings.filterwarnings('ignore')

SEG_SIZE = 256          # длинная сторона вырезки глаза для сегментации и признаков
EPS = 1e-4
CLASSES = ['Норма', 'Лёгкая анемия', 'Умеренная анемия (или тяжелее)']
HB_WORDS = ['Гемоглобин в норме', 'Гемоглобин немного снижен', 'Гемоглобин низкий']
RETAKE = 'Переснять фото'

# ------------------------------------------------------------------ пороги ВОЗ (г/дл)
def who_thresholds(sex, age, pregnant=False):
    """(граница нормы, граница умеренной). Ниже первой — анемия, ниже второй — умеренная и тяжелее."""
    if age is not None and age < 5:
        return 11.0, 10.0
    if pregnant:
        return 11.0, 10.0
    if age is not None and age < 12:
        return 11.5, 11.0
    if age is not None and age < 15:
        return 12.0, 11.0
    return (13.0, 11.0) if sex == 'M' else (12.0, 11.0)

def hb_class(hb, thr_norm, thr_mod):
    return 0 if hb >= thr_norm else 1 if hb >= thr_mod else 2

# ------------------------------------------------------------------ загрузка
def load_rgb(path, max_side=1600):
    """RGB float32 [0,1], с учётом поворота EXIF, поддержка HEIC."""
    from PIL import Image, ImageOps
    try:
        im = Image.open(path)
    except Exception:
        try:
            from pillow_heif import register_heif_opener
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow-heif'], check=False)
            from pillow_heif import register_heif_opener
        register_heif_opener()
        im = Image.open(path)
    im.draft('RGB', (max_side, max_side))          # быстрое JPEG-декодирование в уменьшенном размере
    im = ImageOps.exif_transpose(im).convert('RGB')
    rgb = np.asarray(im).astype(np.float32) / 255
    return resize_max(rgb, max_side)

def resize_max(img, max_side, nearest=False):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    if s >= 1:
        return img
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_AREA
    out = cv2.resize(img.astype(np.uint8) if img.dtype == bool else img, (max(1, round(w * s)), max(1, round(h * s))),
                     interpolation=interp)
    return out > 0 if img.dtype == bool else out

def resize_to(img, shape, nearest=False):
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_AREA
    out = cv2.resize(img.astype(np.uint8) if img.dtype == bool else img, (shape[1], shape[0]), interpolation=interp)
    return out > 0 if img.dtype == bool else out

# ------------------------------------------------------------------ цвет и свет
def srgb_to_lin(x):
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)

def lin_to_srgb(x):
    x = np.clip(x, 0, 1)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)

def relight(rgb, rng, strength=1.0):
    """Симуляция другой лампы и камеры: баланс белого (тёплый/холодный, зелёный/пурпурный), экспозиция, гамма.
    Применяется к обучающим фото, чтобы модель не зависела от освещения, и для стресс-теста."""
    lin = srgb_to_lin(np.clip(rgb, 0, 1))
    t = rng.normal(0, 0.18 * strength)
    tint = rng.normal(0, 0.06 * strength)
    gains = np.exp(np.array([t, tint, -t], np.float32))
    expo = np.exp(rng.normal(0, 0.30 * strength))
    out = lin_to_srgb(lin * gains * expo)
    return (np.clip(out, 0, 1) ** np.exp(rng.normal(0, 0.10 * strength))).astype(np.float32)

def normalize_light(rgb):
    """Для сегментации: «серый мир» (shades of gray, p=6) + выравнивание яркости.
    Признаки для оценки Hb считаются по исходным цветам и по цветам, нормированным по склере."""
    lin = srgb_to_lin(np.clip(rgb, 0, 1)).reshape(-1, 3)
    g = (np.mean(lin ** 6, axis=0) + 1e-12) ** (1 / 6)
    lin = lin * (g.mean() / g)
    v = np.percentile(lin.max(1), 95)
    lin = lin * (0.8 / max(v, 1e-3))
    return lin_to_srgb(lin).reshape(rgb.shape).astype(np.float32)

def white_balance_by(rgb, ref_mask):
    """Эталон белого — склера: делим линейный RGB на средний цвет склеры (модель фон Криса)."""
    lin = srgb_to_lin(np.clip(rgb, 0, 1))
    ref = np.median(lin[ref_mask], axis=0) + 1e-4
    lin = lin * (ref.mean() / ref)
    return lin_to_srgb(lin).astype(np.float32)

# ------------------------------------------------------------------ признаки области (конъюнктива или склера)
def tissue_mask(rgb, mask):
    """Убираем блики (очень светлые и бесцветные), тени и ресницы (тёмные)."""
    v = rgb.max(-1)
    s = (v - rgb.min(-1)) / (v + EPS)
    m = mask & (v > 0.08) & ~((v > 0.92) & (s < 0.25))
    if m.sum() < 30:
        return mask & (v > 0.03)
    lo, hi = np.percentile(v[m], [3, 99])
    return m & (v >= lo) & (v <= hi)

def roi_features(rgb, mask, prefix):
    m = tissue_mask(rgb, mask)
    if m.sum() < 30:
        return None
    px = np.clip(rgb[m].astype(np.float32), 1 / 255, 1)
    R, G, B = px[:, 0], px[:, 1], px[:, 2]
    lab = cv2.cvtColor(px[None], cv2.COLOR_RGB2Lab)[0]
    L, A, Bb = lab[:, 0], lab[:, 1], lab[:, 2]
    hsv = cv2.cvtColor(px[None], cv2.COLOR_RGB2HSV)[0]
    hue = np.where(hsv[:, 0] > 180, hsv[:, 0] - 360, hsv[:, 0])
    S = R + G + B
    ch = {'EI': np.log10(R) - np.log10(G),            # Erythema Index
          'lRB': np.log(R / B), 'lGB': np.log(G / B),
          'r': R / S, 'g': G / S, 'a': A, 'b': Bb, 'L': L, 'hue': hue, 'sat': hsv[:, 1],
          'C': np.hypot(A, Bb), 'hab': np.degrees(np.arctan2(Bb, A))}
    f = {}
    for k, v in ch.items():
        p10, p50, p90 = np.percentile(v, [10, 50, 90])
        f[k + '_m'], f[k + '_p10'], f[k + '_p50'], f[k + '_p90'], f[k + '_iqr'] = \
            float(v.mean()), float(p10), float(p50), float(p90), float(p90 - p10)
    ei = ch['EI']
    f['HHR'] = float(((hue > -10) & (hue < 10)).mean())        # High Hue Ratio
    f['vessel'] = float(np.percentile(ei, 95) - np.median(ei))  # контраст сосудов
    f['a_over_L'] = float(np.median(A) / (np.median(L) + EPS))
    return {prefix + k: v for k, v in f.items()}

# ------------------------------------------------------------------ попиксельные признаки для сегментации
SIGMAS = (0, 2, 4, 8, 16)

def pixel_features(rgb):
    """rgb: вырезка глаза, длинная сторона SEG_SIZE. -> (H, W, F) float32."""
    n = normalize_light(rgb)
    lab = cv2.cvtColor(n, cv2.COLOR_RGB2Lab)
    hsv = cv2.cvtColor(n, cv2.COLOR_RGB2HSV)
    c = np.clip(n, 1 / 255, 1)
    base = [lab[..., 0] / 100, lab[..., 1] / 50, lab[..., 2] / 50, hsv[..., 1],
            np.log(c[..., 0] / c[..., 1]), np.log(c[..., 1] / c[..., 2])]
    feats = []
    for ch in base:
        for s in SIGMAS:
            feats.append(ch if s == 0 else cv2.GaussianBlur(ch, (0, 0), s))
    Lb = lab[..., 0] / 100
    for s in (2, 4):
        b = cv2.GaussianBlur(Lb, (0, 0), s)
        feats.append(np.hypot(cv2.Sobel(b, cv2.CV_32F, 1, 0), cv2.Sobel(b, cv2.CV_32F, 0, 1)))
    a = lab[..., 1] / 50
    feats.append(np.sqrt(np.maximum(cv2.GaussianBlur(a * a, (0, 0), 5) - cv2.GaussianBlur(a, (0, 0), 5) ** 2, 0)))
    h, w = Lb.shape
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    feats += [xx / w, yy / h]
    # положение относительно радужки (в её радиусах): конъюнктива — ниже радужки, склера — по бокам
    iris = find_iris(rgb)
    cx, cy, r = iris if iris is not None else (w / 2, h * 0.4, 0.12 * w)
    feats += [np.clip((xx - cx) / r, -6, 6), np.clip((yy - cy) / r, -6, 6)]
    return np.stack(feats, -1).astype(np.float32)

def _largest_parts(mask, min_frac=0.25):
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if n <= 1:
        return mask
    areas = st[1:, cv2.CC_STAT_AREA]
    keep = 1 + np.where(areas >= areas.max() * min_frac)[0]
    return np.isin(lab, keep)

def segment_conj(rgb, seg_model, thr=0.5):
    """-> (маска конъюнктивы, карта вероятности). rgb — вырезка с длинной стороной SEG_SIZE."""
    F = pixel_features(rgb)
    h, w = F.shape[:2]
    p = seg_model.predict_proba(F.reshape(-1, F.shape[-1]))[:, 1].reshape(h, w).astype(np.float32)
    p = cv2.GaussianBlur(p, (0, 0), 1.5)
    m = p > thr
    m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)) > 0
    m = _largest_parts(m, 0.25)
    if m.sum() < 0.004 * h * w:
        m = np.zeros_like(m)
    return m, p

def find_iris(rgb):
    """Радужка/зрачок: самое тёмное компактное округлое пятно в центральной части кадра -> (cx, cy, r) или None."""
    L = cv2.cvtColor(np.clip(rgb, 0, 1).astype(np.float32), cv2.COLOR_RGB2Lab)[..., 0]
    L = cv2.GaussianBlur(L, (0, 0), 2)
    h, w = L.shape
    best = None
    for q in (6, 10, 15):
        dark = (L < np.percentile(L, q)).astype(np.uint8)
        n, lab, st, cen = cv2.connectedComponentsWithStats(dark, 8)
        for i in range(1, n):
            x, y, bw, bh, a = st[i]
            if a < 0.004 * h * w or bw > 0.5 * w or bh > 0.6 * h:
                continue
            fill = a / (bw * bh)                          # круг заполняет ~0.79 своего квадрата
            aspect = min(bw, bh) / max(bw, bh)
            cx, cy = cen[i]
            central = 1 - min(1, np.hypot((cx - w / 2) / w, (cy - h / 2) / h) * 1.6)
            score = fill * aspect * central * np.sqrt(a)
            if fill > 0.5 and aspect > 0.55 and (best is None or score > best[0]):
                best = (score, cx, cy, max(bw, bh) / 2)
        if best:
            break
    return None if best is None else best[1:]

def find_sclera(rgb, conj):
    """Склера: светлая и самая бесцветная ткань (по сравнению с кожей) по бокам от радужки,
    не конъюнктива, не блик, не фон у края кадра. Ищется по исходным цветам."""
    lab = cv2.cvtColor(np.clip(rgb, 0, 1).astype(np.float32), cv2.COLOR_RGB2Lab)
    L, A, B = lab[..., 0], lab[..., 1], lab[..., 2]
    C = np.hypot(A, B)
    h, w = L.shape
    iris = find_iris(rgb)
    if iris is not None:
        cx, cy, r = iris
        yy, xx = np.mgrid[0:h, 0:w]
        near = (np.abs(yy - cy) < 1.3 * r) & (np.hypot(xx - cx, (yy - cy) * 1.5) < 4.5 * r) & \
               (np.hypot(xx - cx, yy - cy) > 0.9 * r)
        ref_L = np.percentile(L[near], 75)
    else:
        near = np.ones((h, w), bool)
        ref_L = np.percentile(L, 70)
    c_thr = float(np.clip(0.7 * np.median(C), 8, 20))
    cand = near & (C < c_thr) & (L > max(35, ref_L - 12)) & (L < 98)
    if conj.any():
        cand &= ~cv2.dilate(conj.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    cand = cv2.morphologyEx(cand.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)) > 0
    frame = np.zeros((h, w), bool)
    frame[:3], frame[-3:], frame[:, :3], frame[:, -3:] = True, True, True, True
    n_, lab_, st, cen = cv2.connectedComponentsWithStats(cand.astype(np.uint8), 8)
    best = []
    for i in range(1, n_):
        a = st[i, cv2.CC_STAT_AREA]
        if a < 0.002 * h * w or a > 0.25 * h * w:
            continue
        comp = lab_ == i
        if (comp & frame).sum() > 0.02 * a:              # фон/стена/одежда у края кадра — не склера
            continue
        score = a * (1 + (np.median(C) - C[comp].mean()) / 10)
        if conj.any() and iris is None:
            score *= 1.5 if cen[i][1] < np.where(conj)[0].mean() else 0.5   # склера выше оттянутого века
        best.append((score, i))
    if not best:
        return np.zeros((h, w), bool)
    best.sort(reverse=True)
    keep = [i for s_, i in best[:2] if s_ >= best[0][0] * 0.2]
    m = np.isin(lab_, keep)
    return cv2.erode(m.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)

def eye_features(rgb, conj, sclera):
    """Признаки для моделей. c_ — конъюнктива, s_ — склера, w_ — конъюнктива после баланса белого по склере."""
    out = {}
    fc = roi_features(rgb, conj, 'c_') if conj is not None and conj.sum() >= 30 else None
    fs = roi_features(rgb, sclera, 's_') if sclera is not None and sclera.sum() >= 30 else None
    if fc:
        out.update(fc)
    if fs:
        out.update(fs)
    if fc and fs:
        fw = roi_features(white_balance_by(rgb, tissue_mask(rgb, sclera)), conj, 'w_')
        if fw:
            out.update(fw)
    return out

# ------------------------------------------------------------------ поиск глаза на снимке
_FL = None
FL_PATH = 'face_landmarker.task'
R_EYE = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
L_EYE = [263, 249, 390, 373, 374, 380, 381, 382, 362, 398, 384, 385, 386, 387, 388, 466]

def _landmarker():
    global _FL
    if _FL is None:
        try:
            from mediapipe.tasks.python import vision, BaseOptions
            opt = vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=FL_PATH), num_faces=1,
                                               min_face_detection_confidence=0.3, min_face_presence_confidence=0.3)
            _FL = vision.FaceLandmarker.create_from_options(opt)
        except Exception as e:
            print('ℹ️ MediaPipe недоступен, снимок будет считаться крупным планом глаза:', str(e)[:120])
            _FL = False
    return _FL

def eye_candidates(rgb):
    """Список вырезок глаза: по точкам лица (если лицо в кадре) + весь кадр (если это крупный план)."""
    h, w = rgb.shape[:2]
    out = []
    fl = _landmarker()
    if fl:
        try:
            import mediapipe as mp
            small = resize_max(rgb, 1280)
            res = fl.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                     data=np.ascontiguousarray((small * 255).astype(np.uint8))))
            if res.face_landmarks:
                P = np.array([[p.x * w, p.y * h] for p in res.face_landmarks[0]])
                for side, idx in (('правый', R_EYE), ('левый', L_EYE)):
                    e = P[idx]
                    x0, x1 = e[:, 0].min(), e[:, 0].max()
                    cx, cy = (x0 + x1) / 2, e[:, 1].mean()
                    W = max(x1 - x0, 10)
                    box = [int(cx - 0.85 * W), int(cy - 0.55 * W), int(cx + 0.85 * W), int(cy + 0.85 * W)]
                    box = [max(0, box[0]), max(0, box[1]), min(w, box[2]), min(h, box[3])]
                    if box[2] - box[0] > 20 and box[3] - box[1] > 20:
                        out.append({'how': f'лицо, {side} глаз', 'box': box})
        except Exception:
            pass
    out.append({'how': 'крупный план', 'box': [0, 0, w, h]})
    for c in out:
        x0, y0, x1, y1 = c['box']
        c['src_side'] = max(x1 - x0, y1 - y0)
        c['crop'] = resize_max(rgb[y0:y1, x0:x1], SEG_SIZE) if max(x1 - x0, y1 - y0) > SEG_SIZE \
            else cv2.resize(rgb[y0:y1, x0:x1], None, fx=SEG_SIZE / max(x1 - x0, y1 - y0),
                            fy=SEG_SIZE / max(x1 - x0, y1 - y0), interpolation=cv2.INTER_CUBIC)
    return out

# ------------------------------------------------------------------ контроль качества
def quality(crop, conj, sclera, src_side=SEG_SIZE):
    """-> список проблем (пустой — снимок годится). src_side — размер вырезки глаза на исходном фото, px."""
    issues = []
    if src_side < 120:
        issues.append('глаз слишком мелкий в кадре — снимите ближе')
    gray = cv2.cvtColor((np.clip(crop, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    roi = (conj | sclera) if (conj.any() or sclera.any()) else np.ones_like(conj)
    sharp = cv2.Laplacian(gray, cv2.CV_64F)[cv2.dilate(roi.astype(np.uint8), np.ones((9, 9), np.uint8)) > 0].var()
    if sharp < 15 and src_side >= SEG_SIZE:          # увеличенная вырезка всегда «мягкая» — её не браковать
        issues.append('снимок размыт')
    v = crop.max(-1)[roi]
    if np.median(v) < 0.15:
        issues.append('слишком темно')
    if (v > 0.98).mean() > 0.3:
        issues.append('пересвет / вспышка')
    if not conj.any() and not sclera.any():
        issues.append('не видно ни конъюнктивы, ни склеры')
    return issues

print('✅ Ячейка 2: функции готовы')


In [ ]:
#@title ЯЧЕЙКА 2б — ДАННЫЕ CP-AnemiC (признаки цвета 251 фото) { display-mode: "form" }
# Признаки посчитаны функцией roi_features из ячейки 2 по исходным фото + 6 копиям при другом свете; Hb из таблицы датасета.
# Репозиторий публичный, поэтому медицинские данные в нём не хранятся: файл cp_anemic_features.npz
# перетащите в панель «Файлы» слева (или положите в папку моделей на Google Диске) перед запуском.
import base64
CP_FEATS_FILE = 'cp_anemic_features.npz'
_cp_path = next((p for p in (CP_FEATS_FILE, f'{WORK}/{CP_FEATS_FILE}') if os.path.exists(p)), None)
if _cp_path is None:
    try:
        from google.colab import files
        print(f'Загрузите {CP_FEATS_FILE}:')
        files.upload()
        _cp_path = CP_FEATS_FILE if os.path.exists(CP_FEATS_FILE) else None
    except Exception:
        pass
if _cp_path is None:
    raise FileNotFoundError(f'Нет {CP_FEATS_FILE}: перетащите его в «Файлы» и запустите ячейку снова')
CP_FEATS_B64 = base64.b64encode(open(_cp_path, 'rb').read()).decode()
print('✅ Ячейка 2б: признаки CP-AnemiC загружены из', _cp_path)


In [ ]:
#@title ЯЧЕЙКА 3 — ДАННЫЕ, ОБУЧЕНИЕ И ЧЕСТНАЯ ПРОВЕРКА ТОЧНОСТИ { display-mode: "form" }
# Первый запуск: скачивание Eyes-defy-anemia (Kaggle) и обучение ≈3–6 мин. Дальше — загрузка файла моделей за секунды.
# Все цифры точности — на пациентах, которых модель НЕ видела при обучении (кросс-валидация по пациентам).
import io, base64, zipfile
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score

VERSION = 'v11.0'
N_RELIGHT = 3                      # копий каждого фото при другом свете
SEG_PIX = 900                      # пикселей с фото для обучения сегментации

def _clf():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                          l2_regularization=1.0, min_samples_leaf=15, random_state=0)

def _fit(X, y, seed=0):
    """Классы уравниваются повторением редких примеров: модель не может «угадывать» самый частый
    или средний класс. (Повтор вместо sample_weight: в новых scikit-learn веса замедляют обучение в 30 раз.)"""
    rng = np.random.default_rng(seed)
    cls, cnt = np.unique(y, return_counts=True)
    idx = np.concatenate([np.concatenate([np.flatnonzero(y == c),
                                          rng.choice(np.flatnonzero(y == c), cnt.max() - n, replace=True)])
                          for c, n in zip(cls, cnt)])
    return _clf().fit(X[idx], y[idx])

def proba3(m, X):
    """Вероятности трёх классов даже если какого-то класса не было в обучении."""
    P = np.zeros((len(X), 3))
    P[:, m.classes_.astype(int)] = m.predict_proba(X)
    return P

# ------------------------------------------------------------------ Eyes-defy-anemia
def _norm_id(x):
    m = re.search(r'\d+', str(x))
    return str(int(m.group())) if m else None

def _country(p):
    p = p.lower()
    return 'India' if 'india' in p else 'Italy' if 'italy' in p else 'NA'

def _find_col(cols, pats):
    for c in cols:
        if any(re.search(p, str(c).strip().lower()) for p in pats):
            return c
    return None

def eda_index(root):
    recs = []
    for t in glob.glob(f'{root}/**/*', recursive=True):
        if not t.lower().endswith(('.xlsx', '.xls', '.csv')):
            continue
        try:
            sheets = {'csv': pd.read_csv(t)} if t.lower().endswith('.csv') else pd.read_excel(t, sheet_name=None)
        except Exception:
            continue
        for df in sheets.values():
            hb = _find_col(df.columns, [r'^hg?b', r'hgb', r'hemoglob', r'haemoglob', r'emoglob'])
            pid = _find_col(df.columns, [r'^n\.?$', r'^no\.?$', r'^number', r'^id', r'patient', r'numero']) or df.columns[0]
            sex = _find_col(df.columns, [r'sex', r'gender', r'sesso'])
            if hb is None:
                continue
            for _, r in df.iterrows():
                try:
                    hbv = float(str(r[hb]).replace(',', '.'))
                except ValueError:
                    continue
                sx = str(r[sex]).strip().lower()[:1] if sex else ''
                recs.append({'country': _country(t), 'pid': _norm_id(r[pid]), 'Hb': hbv,
                             'sex': 'M' if sx == 'm' else 'F' if sx in ('f', 'w', 'd') else None})
    tab = pd.DataFrame(recs).dropna(subset=['pid', 'Hb'])
    folders = {}
    for p in glob.glob(f'{root}/**/*', recursive=True):
        if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
            folders.setdefault(os.path.dirname(p), []).append(p)
    rows = []
    for folder, files in folders.items():
        low = {f: os.path.basename(f).lower() for f in files}
        palp = [f for f, n in low.items() if 'palpebral' in n and 'forniceal' not in n]
        both = [f for f, n in low.items() if 'palpebral' in n and 'forniceal' in n]
        orig = [f for f, n in low.items() if 'palpebral' not in n and 'forniceal' not in n]
        if (palp or both) and orig:
            rows.append({'country': _country(folder), 'pid': _norm_id(os.path.basename(folder)),
                         'segment': sorted(palp or both)[0], 'original': sorted(orig)[0]})
    imgs = pd.DataFrame(rows)
    keys = ['country', 'pid'] if (tab.country != 'NA').any() else ['pid']
    d = imgs.merge(tab.drop_duplicates(keys), on=keys, how='inner').reset_index(drop=True)
    d['key'] = d.country.astype(str) + '_' + d.pid.astype(str)
    return d

def _mask_of(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        return None
    if im.ndim == 2:
        return im > 15
    if im.shape[2] == 4:
        return im[..., 3] > 0
    return im[..., :3].max(-1) > 15

def _prep_one(row):
    """Фото -> вырезка (весь кадр, как «крупный план» в ячейке 4) + маска врача + попиксельные признаки."""
    try:
        rgb = load_rgb(row['original'], 1024)
        crop = resize_max(rgb, SEG_SIZE)
        m = _mask_of(row['segment'])
        if m is None or abs(m.shape[0] / m.shape[1] - rgb.shape[0] / rgb.shape[1]) > 0.02:
            return None
        doc = resize_to(m, crop.shape[:2], nearest=True)
        if doc.sum() < 30:
            return None
        return {'key': row['key'], 'crop': crop, 'doc': doc, 'F': pixel_features(crop)}
    except Exception:
        return None

def _sample_pixels(item, rng):
    F, doc = item['F'], item['doc']
    ring = cv2.dilate(doc.astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool) & ~doc
    pos, near, far = np.flatnonzero(doc), np.flatnonzero(ring), np.flatnonzero(~doc & ~ring)
    n = SEG_PIX // 3
    idx = np.concatenate([rng.choice(a, min(n, len(a)), replace=False) for a in (pos, near, far) if len(a)])
    Ff = F.reshape(-1, F.shape[-1])
    return Ff[idx], doc.reshape(-1)[idx].astype(int)

def _train_seg(items, seed=0):
    rng = np.random.default_rng(seed)
    XY = [_sample_pixels(it, rng) for it in items]
    X, y = np.concatenate([a for a, _ in XY]), np.concatenate([b for _, b in XY])
    m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, max_iter=150, random_state=seed)
    return m.fit(X, y)

def _predict_mask(item, seg):
    F = item['F']
    h, w = F.shape[:2]
    p = seg.predict_proba(F.reshape(-1, F.shape[-1]))[:, 1].reshape(h, w).astype(np.float32)
    p = cv2.GaussianBlur(p, (0, 0), 1.5)
    m = p > 0.5
    m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)) > 0
    m = _largest_parts(m, 0.25)
    return m if m.sum() >= 0.004 * h * w else np.zeros_like(m)

def _dice(a, b):
    return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

def _feat_rows(key, crop, conj, sclera, seed):
    rows = [{'key': key, 'aug': 0, **eye_features(crop, conj, sclera)}]
    rng = np.random.default_rng(seed)
    for a in range(1, N_RELIGHT + 1):
        rows.append({'key': key, 'aug': a, **eye_features(relight(crop, rng), conj, sclera)})
    return rows

# ------------------------------------------------------------------ CV одной модели
def cv_model(Fd, feats, y_of, groups_of, reps=2):
    """Fd: таблица признаков (aug 0 — исходные фото, aug>0 — другой свет). Возвращает OOF-вероятности и метрики."""
    O = Fd[Fd.aug == 0].reset_index(drop=True)
    y = O.key.map(y_of).values
    P = np.zeros((reps, len(O), 3))
    PA, YA = [], []
    for rep in range(reps):
        skf = StratifiedKFold(5, shuffle=True, random_state=rep)
        for tr, te in skf.split(O, y):
            trk, tek = set(O.key.values[tr]), set(O.key.values[te])
            T = Fd[Fd.key.isin(trk)]
            m = _fit(T[feats].values, T.key.map(y_of).values)
            P[rep, te] = proba3(m, O[feats].values[te])
            if rep == 0:
                A = Fd[Fd.key.isin(tek) & (Fd.aug > 0)]
                PA.append(proba3(m, A[feats].values)); YA.append(A.key.map(y_of).values)
    PA, YA = np.concatenate(PA), np.concatenate(YA)
    ba = [balanced_accuracy_score(y, P[r].argmax(1)) for r in range(reps)]
    try:
        auc = roc_auc_score(y > 0, 1 - P.mean(0)[:, 0])
        auc_l = roc_auc_score(YA > 0, 1 - PA[:, 0])
    except ValueError:
        auc = auc_l = float('nan')
    cm = confusion_matrix(y, P[0].argmax(1), labels=[0, 1, 2])
    return {'keys': O.key.values, 'y': y, 'P': P.mean(0), 'BA': float(np.mean(ba)), 'BA_light': float(
        balanced_accuracy_score(YA, PA.argmax(1))), 'AUC': float(auc), 'AUC_light': float(auc_l),
        'cm': cm, 'recall': (cm.diagonal() / np.maximum(cm.sum(1), 1)), 'n': len(O),
        'pred_counts': np.bincount(P[0].argmax(1), minlength=3)}

def print_cv(name, r):
    rec = ' / '.join(f'{v * 100:.0f}%' for v in r['recall'])
    print(f"   {name:<34} n={r['n']:<4} верный класс (сбаланс.) {r['BA'] * 100:4.0f}%  "
          f"| при другом свете {r['BA_light'] * 100:4.0f}%  | AUC анемия/нет {r['AUC']:.2f} "
          f"(другой свет {r['AUC_light']:.2f})  | найдено по классам N/Л/У: {rec}")

# ------------------------------------------------------------------ CP-AnemiC (дети до 5 лет): признаки зашиты в ноутбук
def load_cp_features():
    z = np.load(io.BytesIO(base64.b64decode(CP_FEATS_B64)), allow_pickle=False)
    df = pd.DataFrame(z['X'], columns=[str(c) for c in z['cols']])
    df['key'] = ['CP_' + str(int(i)) for i in z['idx']]
    df['aug'] = z['aug'].astype(int)
    hb = dict(zip(['CP_' + str(int(i)) for i in z['hb_idx']], z['hb']))
    return df, hb

# ------------------------------------------------------------------ обучение всего
def train_all():
    t0 = time.time()
    report = {}
    # ---- 1. дети до 5 лет (CP-AnemiC, телефон, Гана)
    cp, cp_hb = load_cp_features()
    cp_y = {k: hb_class(v, 11.0, 10.0) for k, v in cp_hb.items()}
    cpf = [c for c in cp.columns if c.startswith('c_')]
    print(f'1/4 CP-AnemiC (дети до 5 лет): {len(cp_hb)} фото, классы N/Л/У = '
          f'{np.bincount(list(cp_y.values()), minlength=3).tolist()}')
    r = cv_model(cp, cpf, cp_y, None)
    print_cv('Дети <5, конъюнктива', r)
    report['child_conj'] = r
    models = {'child_conj': (_fit(cp[cpf].values, cp.key.map(cp_y).values), cpf)}

    # ---- 2. взрослые (Eyes-defy-anemia): загрузка
    print(f'2/4 Eyes-defy-anemia: скачивание и подготовка фото... ({time.time() - t0:.0f} с)')
    import kagglehub
    root = kagglehub.dataset_download('harshwardhanfartale/eyes-defy-anemia')
    d = eda_index(root)
    if len(d) == 0:
        raise RuntimeError('В скачанном Eyes-defy-anemia не найдены фото с разметкой и Hb')
    items = [x for x in Parallel(n_jobs=-1)(delayed(_prep_one)(r) for r in d.to_dict('records')) if x]
    d = d[d.key.isin({it['key'] for it in items})].reset_index(drop=True)
    sex = dict(zip(d.key, d.sex))
    hbv = dict(zip(d.key, d.Hb))
    ad_y = {k: hb_class(hbv[k], *who_thresholds(sex[k], 30)) for k in d.key}
    print(f'   {len(items)} пациентов, классы N/Л/У = {np.bincount(list(ad_y.values()), minlength=3).tolist()} '
          f'({time.time() - t0:.0f} с)')

    # ---- 3. сегментация конъюнктивы: проверка на невиданных пациентах + финальная модель
    print(f'3/4 Обучение поиска конъюнктивы по разметке врачей... ({time.time() - t0:.0f} с)')
    oof = {}
    for tr, te in GroupKFold(5).split(items, groups=[it['key'] for it in items]):
        seg = _train_seg([items[i] for i in tr])
        for i in te:
            oof[items[i]['key']] = _predict_mask(items[i], seg)
    dice = np.array([_dice(oof[it['key']], it['doc']) for it in items])
    found = np.array([oof[it['key']].any() for it in items])
    print(f'   Конъюнктива найдена у {found.sum()}/{len(items)}; совпадение с разметкой врачей (Dice), медиана '
          f'{np.median(dice) * 100:.0f}%, у четверти худших ниже {np.percentile(dice, 25) * 100:.0f}%  '
          f'({time.time() - t0:.0f} с)')
    report['seg'] = {'dice': dice, 'found': found}
    seg_final = _train_seg(items, seed=1)

    # ---- 4. признаки (маски из проверки на невиданных пациентах — как будет на новом фото) и модели
    print(f'4/4 Признаки цвета и модели... ({time.time() - t0:.0f} с)')
    rows = Parallel(n_jobs=-1)(delayed(_feat_rows)(
        it['key'], it['crop'], oof[it['key']], find_sclera(it['crop'], oof[it['key']]), i)
        for i, it in enumerate(items))
    Fd = pd.DataFrame([r for rr in rows for r in rr])
    Fd['male'] = Fd.key.map(lambda k: 1.0 if sex[k] == 'M' else 0.0 if sex[k] == 'F' else 0.5)
    sets = {'conj': [c for c in Fd.columns if c.startswith('c_')] + ['male'],
            'sclera': [c for c in Fd.columns if c.startswith('s_')] + ['male']}
    sets['both'] = sets['conj'][:-1] + sets['sclera'][:-1] + [c for c in Fd.columns if c.startswith('w_')] + ['male']
    names = {'conj': 'Взрослые, только конъюнктива', 'sclera': 'Взрослые, только склера',
             'both': 'Взрослые, конъюнктива + склера'}
    for s, feats in sets.items():
        sub = Fd.dropna(subset=feats)
        if sub.key.nunique() < 40:
            print(f'   {names[s]}: мало фото ({sub.key.nunique()}), модель не обучается')
            continue
        r = cv_model(sub, feats, ad_y, None)
        print_cv(names[s], r)
        report[s] = r
        models[s] = (_fit(sub[feats].values, sub.key.map(ad_y).values), feats)

    # порядок предпочтения методов — по измеренной точности, а не по договорённости
    order = sorted([s for s in ('both', 'conj', 'sclera') if s in report], key=lambda s: -report[s]['BA'])
    print('   Порядок выбора метода (по проверенной точности):', ' > '.join(names[s] for s in order))

    # итог всей системы целиком: на каждом фото берётся лучший доступный метод
    O = Fd[Fd.aug == 0].set_index('key')
    final_y, final_p = [], []
    for k in O.index:
        for s in order:
            r = report[s]
            if k in set(r['keys']):
                final_y.append(ad_y[k]); final_p.append(r['P'][list(r['keys']).index(k)].argmax())
                break
        else:
            final_y.append(ad_y[k]); final_p.append(3)
    cm = confusion_matrix(final_y, final_p, labels=[0, 1, 2, 3])[:3]
    report['system'] = {'cm': cm, 'y': np.array(final_y), 'p': np.array(final_p)}
    ok = np.array(final_p) < 3
    print(f'\n   ВСЯ СИСТЕМА (взрослые, {len(final_y)} пациентов): верный класс (сбаланс.) '
          f'{balanced_accuracy_score(np.array(final_y)[ok], np.array(final_p)[ok]) * 100:.0f}% '
          f'на {ok.sum()} фото; «переснять» у {(~ok).sum()}')
    bundle = {'version': VERSION, 'sklearn': sklearn.__version__, 'seg': seg_final, 'models': models,
              'order': order, 'names': names,
              'report': {k: {kk: vv for kk, vv in v.items() if kk not in ('keys', 'P')} for k, v in report.items()},
              'trained_s': time.time() - t0}
    print(f'\n✅ Обучение завершено за {time.time() - t0:.0f} с')
    return bundle

# ------------------------------------------------------------------ загрузить или обучить
T0 = time.time()
BUNDLE = None
if os.path.exists(MODEL_PATH):
    try:
        b = joblib.load(MODEL_PATH)
        if b.get('version') == VERSION and b.get('sklearn') == sklearn.__version__:
            BUNDLE = b
        else:
            print('ℹ️ Файл моделей от другой версии — обучаю заново.')
    except Exception as e:
        print('ℹ️ Файл моделей не читается — обучаю заново:', str(e)[:100])
if BUNDLE is None:
    BUNDLE = train_all()
    joblib.dump(BUNDLE, MODEL_PATH, compress=3)
    if not ON_DRIVE:
        try:
            from google.colab import files
            files.download(MODEL_PATH)
            print(f'💾 {MODEL_NAME} скачивается на компьютер: в следующий раз перетащите его в «Файлы» — обучение не понадобится.')
        except Exception:
            pass
else:
    print(f'✅ Модели загружены за {time.time() - T0:.1f} с (обучены ранее за {BUNDLE["trained_s"]:.0f} с)')
SEG_MODEL = BUNDLE['seg']


## 📷 Анализ фото
Заполните пол и возраст справа от кода и запустите ячейку 4. Снимайте при дневном свете, без вспышки, нижнее веко оттянуто.

In [ ]:
#@title 📷 ЯЧЕЙКА 4 — АНАЛИЗ ФОТО (заполните поля и запустите) { display-mode: "form" }
#@markdown Сфотографируйте глаз с **оттянутым нижним веком** при дневном свете, без вспышки и фильтров.
#@markdown Можно 1–3 снимка: результат усредняется. Лицо целиком или крупный план — оба варианта подходят.
Пол = 'Женский'  #@param ["Женский", "Мужской"]
Возраст = 20  #@param {type:"integer"}
Беременность = False  #@param {type:"boolean"}
#@markdown *(Необязательно)* путь к фото уже в Colab, например `/content/eye.jpg`. Пусто — откроется окно выбора файлов.
Путь_к_фото = ''  #@param {type:"string"}

def analyze_photo(path, sex, age):
    t0 = time.time()
    rgb = load_rgb(path, 1600)
    cands = []
    for c in eye_candidates(rgb):
        conj, p = segment_conj(c['crop'], SEG_MODEL)
        # уверенность × размер найденной конъюнктивы; вырезка по точкам лица важнее всего кадра
        score = float(p[conj].mean() * np.sqrt(conj.mean())) if conj.any() else 0.0
        cands.append({**c, 'conj': conj, 'prob': p, 'score': score})
    face = [c for c in cands if c['how'] != 'крупный план']
    pool = [c for c in face if c['score'] > 0] or [c for c in cands if c['score'] > 0] or face or cands
    best = max(pool, key=lambda c: c['score'])
    crop, conj = best['crop'], best['conj']
    sclera = find_sclera(crop, conj)
    issues = quality(crop, conj, sclera, best['src_side'])
    feats = eye_features(crop, conj, sclera)
    feats['male'] = 1.0 if sex == 'M' else 0.0
    child = age is not None and age < 5
    order = ['child_conj'] if child else BUNDLE['order']
    method, P = None, None
    if not any(i.startswith(('снимок размыт', 'слишком темно', 'пересвет', 'глаз слишком мелкий')) for i in issues):
        for s in order:
            if s not in BUNDLE['models']:
                continue
            m, cols = BUNDLE['models'][s]
            if all(c in feats and np.isfinite(feats[c]) for c in cols):
                P = proba3(m, np.array([[feats[c] for c in cols]]))[0]
                method = s
                break
    return {'path': path, 'how': best['how'], 'crop': crop, 'conj': conj, 'sclera': sclera, 'issues': issues,
            'method': method, 'P': P, 'seconds': time.time() - t0}

def method_reason(r):
    has_c, has_s = r['conj'].any(), r['sclera'].any()
    if r['method'] == 'both':
        return 'видны и конъюнктива, и склера — методы объединены (цвет века нормирован по белку глаза)'
    if r['method'] == 'conj':
        return ('склера не найдена — оценка только по конъюнктиве' if not has_s else
                'по проверке на пациентах конъюнктива одна точнее, чем вместе со склерой')
    if r['method'] == 'sclera':
        return ('конъюнктива не видна — оценка только по склере (менее точно)' if not has_c else
                'по проверке на пациентах склера здесь точнее')
    if r['method'] == 'child_conj':
        return 'ребёнок до 5 лет — модель CP-AnemiC по конъюнктиве'
    return ''

def show(results, sex, age, pregnant):
    ok = [r for r in results if r['P'] is not None]
    n = len(results)
    fig, ax = plt.subplots(1, n + 1, figsize=(4.2 * (n + 1), 4))
    ax = np.atleast_1d(ax)
    for a, r in zip(ax, results):
        vis = (np.clip(r['crop'], 0, 1) * 255).astype(np.uint8).copy()
        for m, col in ((r['conj'], (0, 255, 0)), (r['sclera'], (0, 160, 255))):
            cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(vis, cnts, -1, col, 2)
        a.imshow(vis); a.axis('off')
        a.set_title(f"{os.path.basename(r['path'])[:22]}\nзелёный — конъюнктива, голубой — склера", fontsize=9)
    out = {'version': VERSION, 'sex': sex, 'age': age, 'pregnant': pregnant, 'photos': []}
    for r in results:
        out['photos'].append({'file': os.path.basename(r['path']), 'eye': r['how'], 'method': r['method'],
                              'issues': r['issues'], 'seconds': round(r['seconds'], 2),
                              'probabilities': None if r['P'] is None else [round(float(v), 3) for v in r['P']]})
    if ok:
        P = np.mean([r['P'] for r in ok], axis=0)
        k = int(P.argmax())
        ax[-1].barh(CLASSES[::-1], P[::-1] * 100, color=['#c0392b', '#e67e22', '#27ae60'])
        ax[-1].set_xlim(0, 100); ax[-1].set_xlabel('вероятность, %'); ax[-1].set_title('Шкала риска')
        out.update({'result': CLASSES[k], 'hb': HB_WORDS[k], 'probabilities': dict(zip(CLASSES, map(float, P)))})
    else:
        ax[-1].axis('off')
        out.update({'result': RETAKE, 'hb': None})
    plt.tight_layout(); plt.show()

    print('=' * 78)
    if ok:
        conf = P.max()
        print(f"  РЕЗУЛЬТАТ: {out['result'].upper()}   ·   {out['hb']}")
        print('  ' + '   '.join(f'{c}: {p * 100:.0f}%' for c, p in zip(CLASSES, P)))
        if conf < 0.5:
            second = CLASSES[int(np.argsort(P)[-2])]
            print(f'  ⚠️ Уверенность низкая ({conf * 100:.0f}%): результат между «{out["result"]}» и «{second}». '
                  'Переснимите при дневном свете или сдайте анализ крови.')
        if Беременность or (age is not None and 5 <= age < 15):
            print('  ℹ️ Модель обучена на взрослых (небеременных). Для беременных и детей 5–14 лет граница нормы по ВОЗ '
                  f'другая: {who_thresholds(sex, age, pregnant)[0]:.1f} г/дл — сверяйте с анализом крови.')
        for r in ok:
            rep = BUNDLE['report'].get(r['method'], {})
            print(f"  Метод ({os.path.basename(r['path'])[:20]}): {BUNDLE['names'].get(r['method'], 'конъюнктива, дети')}"
                  f" — {method_reason(r)}. Проверенная точность метода: верный класс {rep.get('BA', np.nan) * 100:.0f}%,"
                  f" AUC {rep.get('AUC', np.nan):.2f}")
    else:
        print(f"  РЕЗУЛЬТАТ: {RETAKE.upper()}")
    for r in results:
        if r['P'] is None or r['issues']:
            why = ', '.join(r['issues']) or 'конъюнктива для ребёнка до 5 лет не найдена'
            print(f"  📷 {os.path.basename(r['path'])[:24]}: {why}. Совет: дневной свет у окна, без вспышки, "
                  'нижнее веко оттянуто, глаз в фокусе.')
    print(f"  Время анализа: {sum(r['seconds'] for r in results):.1f} с на {n} фото")
    print('  ⚠️ Это скрининг, а не диагноз. Точный уровень гемоглобина показывает только анализ крови.')
    print('=' * 78)
    os.makedirs('results', exist_ok=True)
    with open('results/result.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    return out

sex = 'M' if Пол == 'Мужской' else 'F'
paths = [p.strip() for p in Путь_к_фото.split(',') if p.strip()]
if not paths:
    from google.colab import files
    print('Выберите 1–3 фото глаза с оттянутым нижним веком:')
    paths = list(files.upload().keys())
results = [analyze_photo(p, sex, Возраст) for p in paths[:3]]
RESULT = show(results, sex, Возраст, Беременность)


In [ ]:
#@title ЯЧЕЙКА 5 — (для защиты) ТОЧНОСТЬ: матрицы ошибок на пациентах, которых модель не видела { display-mode: "form" }
rep = BUNDLE['report']
names = {**BUNDLE['names'], 'child_conj': 'Дети <5 (CP-AnemiC), конъюнктива'}
keys = [k for k in ('both', 'conj', 'sclera', 'child_conj') if k in rep]
fig, ax = plt.subplots(1, len(keys) + 1, figsize=(4.3 * (len(keys) + 1), 4))
short = ['Норма', 'Лёгкая', 'Умер.+']
for a, k in zip(ax, keys):
    cm = rep[k]['cm']
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    a.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            a.text(j, i, f'{cm[i, j]}\n{cmn[i, j] * 100:.0f}%', ha='center', va='center',
                   color='white' if cmn[i, j] > 0.6 else 'black', fontsize=9)
    a.set_xticks(range(3)); a.set_xticklabels(short); a.set_yticks(range(3)); a.set_yticklabels(short)
    a.set_xlabel('ответ системы'); a.set_ylabel('по анализу крови')
    a.set_title(f"{names[k]}\nверный класс {rep[k]['BA'] * 100:.0f}% · AUC {rep[k]['AUC']:.2f}", fontsize=9)
if 'seg' in rep:
    ax[-1].hist(rep['seg']['dice'] * 100, bins=20, color='#2e86c1')
    ax[-1].set_title(f"Поиск конъюнктивы: совпадение с врачами\nмедиана {np.median(rep['seg']['dice']) * 100:.0f}%",
                     fontsize=9)
    ax[-1].set_xlabel('Dice, %')
plt.tight_layout(); plt.show()
print('Как читать: строки — настоящий класс по анализу крови, столбцы — ответ системы. Диагональ — верные ответы.')
print('«Верный класс» — сбалансированная точность: каждый класс весит одинаково, поэтому система не может')
print('набрать балл, называя всех «нормой» или «лёгкой». Случайное угадывание из 3 классов дало бы 33%.')
print('«При другом свете» — те же пациенты, перекрашенные под тёплую/холодную лампу, другую экспозицию и камеру.')
for k in keys:
    print(f"  {names[k]:<36} предсказано по классам N/Л/У: {[int(v) for v in rep[k]['pred_counts']]}, "
          f"на самом деле: {[int(v) for v in np.bincount(rep[k]['y'], minlength=3)]}")
